In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

sns.set_style('whitegrid')
WORK = '/kaggle/working'

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]

BASE = os.path.dirname(find('train.csv'))
test = pd.read_csv(f'{BASE}/test.csv')
folds = pd.read_csv(find('folds.csv'))
w_uids = pd.read_csv(find('wavlm_uids.csv')).uid.tolist()
Em = np.load(find('wavlm_large_mean.npy'))
Es = np.load(find('wavlm_large_std.npy'))
print(Em.shape, Es.shape, len(w_uids), folds.shape)

idx = {u: i for i, u in enumerate(w_uids)}
rows_tr = np.array([idx[u] for u in folds.uid])
rows_te = np.array([idx['test/' + f] for f in test.filename])

y = folds.label.values
w = folds.w_test.values
nz = (folds.is_zero == 0).values
grp = folds.dur_group.values

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def pear(a, b): return float(np.corrcoef(a, b)[0, 1])
def wrmse(a, b, ww): return float(np.sqrt(np.sum(ww * (a - b) ** 2) / np.sum(ww)))
def wpear(a, b, ww):
    ma, mb = np.average(a, weights=ww), np.average(b, weights=ww)
    return float(np.sum(ww * (a - ma) * (b - mb)) /
                 np.sqrt(np.sum(ww * (a - ma) ** 2) * np.sum(ww * (b - mb) ** 2)))

def report(oof):
    r = {'non_zero': (rmse(y[nz], oof[nz]), pear(y[nz], oof[nz]))}
    for g in ['dur_45s', 'dur_60s']:
        m = nz & (grp == g)
        r[g] = (rmse(y[m], oof[m]), pear(y[m], oof[m]))
    r['test_weighted'] = (wrmse(y[nz], oof[nz], w[nz]), wpear(y[nz], oof[nz], w[nz]))
    return pd.DataFrame(r, index=['RMSE', 'Pearson']).T.round(4)

def ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 5, 9)))

def cv_oof(X, make_model=ridge, rep='rep0'):
    oof = np.full(len(y), np.nan)
    for f in range(5):
        va = (folds[rep] == f).values
        m = make_model()
        m.fit(X[(~va) & nz], y[(~va) & nz])
        oof[va & nz] = np.clip(m.predict(X[va & nz]), 1, 5)
    return oof

In [ ]:
res = []
for L in range(Em.shape[1]):
    r = report(cv_oof(Em[rows_tr, L].astype(np.float32)))
    res.append(dict(layer=L,
                    rmse_w=r.loc['test_weighted', 'RMSE'], pear_w=r.loc['test_weighted', 'Pearson'],
                    rmse_45=r.loc['dur_45s', 'RMSE'], pear_45=r.loc['dur_45s', 'Pearson'],
                    rmse_60=r.loc['dur_60s', 'RMSE'], pear_60=r.loc['dur_60s', 'Pearson']))
res = pd.DataFrame(res).round(4)
print(res.sort_values('pear_w', ascending=False).head(8).to_string(index=False))
print('\nreference: dummy test-weighted RMSE 0.9675, Pearson about 0')

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(res.layer, res.pear_w, marker='o', label='test-weighted')
ax[0].plot(res.layer, res.pear_45, marker='.', label='45s')
ax[0].plot(res.layer, res.pear_60, marker='.', label='60s')
ax[0].set_title('CV Pearson by WavLM layer'); ax[0].set_xlabel('layer'); ax[0].legend()
ax[1].plot(res.layer, res.rmse_w, marker='o')
ax[1].axhline(0.9675, color='gray', ls='--')
ax[1].set_title('Test-weighted CV RMSE by layer (dashed = dummy)'); ax[1].set_xlabel('layer')
plt.tight_layout(); plt.show()

In [ ]:
b = int(res.sort_values('pear_w').iloc[-1].layer)
print('best layer by test-weighted Pearson:', b)

def make_X(kind, rows):
    if kind == 'mean': return Em[rows, b].astype(np.float32)
    if kind == 'std': return Es[rows, b].astype(np.float32)
    if kind == 'mean+std': return np.hstack([Em[rows, b], Es[rows, b]]).astype(np.float32)
    if kind == 'mean_win5': return Em[rows, max(b - 2, 0):b + 3].astype(np.float32).mean(1)

out = []
for kind in ['mean', 'std', 'mean+std', 'mean_win5']:
    r = report(cv_oof(make_X(kind, rows_tr)))
    out.append(dict(kind=kind,
                    pear_w=r.loc['test_weighted', 'Pearson'], rmse_w=r.loc['test_weighted', 'RMSE'],
                    pear_45=r.loc['dur_45s', 'Pearson'], pear_60=r.loc['dur_60s', 'Pearson']))
out = pd.DataFrame(out)
print(out.round(4).to_string(index=False))
best_kind = out.sort_values('pear_w').iloc[-1].kind
print('chosen:', best_kind)

oof = cv_oof(make_X(best_kind, rows_tr))
print(report(oof))
model = ridge().fit(make_X(best_kind, rows_tr)[nz], y[nz])
pred_tr = np.clip(model.predict(make_X(best_kind, rows_tr)), 1, 5)
pred_te = np.clip(model.predict(make_X(best_kind, rows_te)), 1, 5)
print('TRAIN RMSE (in-sample, non-zero):', round(rmse(y[nz], pred_tr[nz]), 4))
print('test prediction summary:', pd.Series(pred_te).describe().round(3).to_dict())

pd.DataFrame({'uid': folds.uid, 'oof': oof}).to_csv(f'{WORK}/oof_wavlm.csv', index=False)
pd.DataFrame({'filename': test.filename, 'pred': pred_te}).to_csv(f'{WORK}/test_wavlm.csv', index=False)
pd.DataFrame({'filename': test.filename, 'label': pred_te}).to_csv(f'{WORK}/submission_wavlm.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(x=y[nz], y=oof[nz], hue=grp[nz], alpha=0.5, ax=ax[0])
ax[0].plot([1, 5], [1, 5], 'r--'); ax[0].set_xlabel('true'); ax[0].set_ylabel('OOF prediction')
ax[0].set_title('WavLM: predicted vs true')
sns.histplot(oof[nz] - y[nz], bins=30, ax=ax[1]); ax[1].set_title('Residuals')
plt.tight_layout(); plt.show()

In [ ]:
from scipy.stats import ks_2samp, spearmanr

meta = pd.read_csv(find('audio_meta.csv'))
tdur = meta[meta.split == 'test'].set_index('filename').duration.reindex(test.filename).values
test_grp = np.where(tdur < 40, 'short', np.where(tdur < 52, 'dur_45s', 'dur_60s'))

def l2n(Z):
    return Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)

rows_out, keep = [], {}
for L in [0, 4, 8, 21]:
    Xtr = Em[rows_tr, L].astype(np.float32)
    Xte = Em[rows_te, L].astype(np.float32)
    cv_sim = np.full(len(y), np.nan)
    cv_nnlab = np.full(len(y), np.nan)
    te_sim = np.zeros((5, len(Xte)))
    for f in range(5):
        va = (folds.rep0 == f).values
        ref, qry = (~va) & nz, va & nz
        sc = StandardScaler().fit(Xtr[ref])
        R, Q, T = (l2n(sc.transform(a)) for a in (Xtr[ref], Xtr[qry], Xte))
        S = Q @ R.T
        cv_sim[qry] = S.max(1)
        cv_nnlab[qry] = y[ref][S.argmax(1)]
        te_sim[f] = (T @ R.T).max(1)
    te_m = te_sim.mean(0)
    keep[L] = (cv_sim, te_m)
    for g in ['dur_45s', 'dur_60s']:
        m = nz & (grp == g)
        a, b = cv_sim[m], te_m[test_grp == g]
        rows_out.append(dict(layer=L, group=g, nn_sim_cv=a.mean(), nn_sim_test=b.mean(),
                             ks_p=ks_2samp(a, b).pvalue,
                             nn_label_rho=spearmanr(cv_nnlab[m], y[m])[0]))
print(pd.DataFrame(rows_out).round(4).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a_, L in zip(ax, [4, 21]):
    cv_sim, te_m = keep[L]
    m = nz & (grp == 'dur_45s')
    sns.kdeplot(cv_sim[m], label='CV validation (45s)', ax=a_)
    sns.kdeplot(te_m[test_grp == 'dur_45s'], label='test (45s)', ax=a_)
    a_.set_title(f'Nearest-neighbour similarity to training set, layer {L}')
    a_.legend()
plt.tight_layout(); plt.show()

print('\nstability across the 3 CV repeats (layer 21, mean pooling):')
for rep in ['rep0', 'rep1', 'rep2']:
    r = report(cv_oof(Em[rows_tr, 21].astype(np.float32), rep=rep))
    print(rep, 'test-weighted:', r.loc['test_weighted'].to_dict(), '| 45s:', r.loc['dur_45s'].to_dict())

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
rng = np.random.RandomState(SEED)

# speaker/channel-like representation: early layers, mean + std
def grp_feats(rows):
    return np.hstack([Em[rows, L] for L in (0, 2, 4)] + [Es[rows, L] for L in (0, 2, 4)]).astype(np.float32)

G_tr, G_te = grp_feats(rows_tr), grp_feats(rows_te)
sc_g = StandardScaler().fit(G_tr[nz])
pca = PCA(n_components=50, random_state=SEED).fit(sc_g.transform(G_tr[nz]))
Z_tr = pca.transform(sc_g.transform(G_tr))

X4_tr = Em[rows_tr, 4].astype(np.float32)
X4_te = Em[rows_te, 4].astype(np.float32)

def nn_sim_stats(col):
    val_sim = np.full(len(y), np.nan)
    te_sim = np.zeros((5, len(X4_te)))
    for f in range(5):
        va = (folds[col] == f).values
        ref, qry = (~va) & nz, va & nz
        sc = StandardScaler().fit(X4_tr[ref])
        R, Q, T = (l2n(sc.transform(a)) for a in (X4_tr[ref], X4_tr[qry], X4_te))
        val_sim[qry] = (Q @ R.T).max(1)
        te_sim[f] = (T @ R.T).max(1)
    return val_sim, te_sim.mean(0)

def summarise(col, name, extra=None):
    val_sim, te_m = nn_sim_stats(col)
    r = report(cv_oof(Em[rows_tr, 21].astype(np.float32), rep=col))
    row = dict(folds=name)
    if extra:
        row.update(extra)
    for g in ['dur_45s', 'dur_60s']:
        k = g[4:6]
        row[f'sim_val_{k}'] = val_sim[nz & (grp == g)].mean()
        row[f'sim_test_{k}'] = te_m[test_grp == g].mean()
    row.update(pear_w=r.loc['test_weighted', 'Pearson'], rmse_w=r.loc['test_weighted', 'RMSE'],
               pear_45=r.loc['dur_45s', 'Pearson'], pear_60=r.loc['dur_60s', 'Pearson'])
    return row

ybin = np.digitize(y, [2.25, 3.25, 4.25])
strat_lab = ybin * 2 + (grp == 'dur_60s').astype(int)
idx_nz = np.where(nz)[0]

table = [summarise('rep0', 'random (rep0)')]
for k in [30, 60, 120, 250]:
    cl = AgglomerativeClustering(n_clusters=k, metric='cosine', linkage='average').fit_predict(Z_tr[nz])
    col = f'g{k}'
    folds[col] = rng.randint(0, 5, len(y))      # zero-label rows get arbitrary folds, never used
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for f, (_, va_local) in enumerate(sgkf.split(idx_nz, strat_lab[nz], groups=cl)):
        folds.loc[idx_nz[va_local], col] = f
    fs = folds.loc[nz, col].value_counts()
    table.append(summarise(col, col, dict(biggest_cluster=int(np.bincount(cl).max()),
                                          fold_min=int(fs.min()), fold_max=int(fs.max()))))

T = pd.DataFrame(table).round(4)
pd.set_option('display.width', 220)
print(T.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(T.folds, T.pear_w, marker='o', label='test-weighted')
ax[0].plot(T.folds, T.pear_45, marker='.', label='45s')
ax[0].plot(T.folds, T.pear_60, marker='.', label='60s')
ax[0].set_title('CV Pearson (layer 21) by fold scheme'); ax[0].legend()
ax[1].plot(T.folds, T.sim_val_60, marker='o', label='CV val (60s)')
ax[1].axhline(T.sim_test_60.mean(), color='gray', ls='--', label='test (60s)')
ax[1].plot(T.folds, T.sim_val_45, marker='o', label='CV val (45s)')
ax[1].axhline(T.sim_test_45.mean(), color='black', ls=':', label='test (45s)')
ax[1].set_title('Nearest-neighbour similarity, layer 4'); ax[1].legend()
plt.tight_layout(); plt.show()

folds[['uid', 'rep0', 'rep1', 'rep2', 'g30', 'g60', 'g120', 'g250']].to_csv(f'{WORK}/folds_group.csv', index=False)

In [ ]:
sub = pd.read_csv(f'{WORK}/submission_wavlm.csv')
print(sub.shape, sub.columns.tolist())
print('NaN:', int(sub.label.isna().sum()), '| unique filenames:', sub.filename.nunique())
print(sub.head(3))

In [ ]:
rows = []
for col in ['g250', 'g120']:
    for L in range(Em.shape[1]):
        r = report(cv_oof(Em[rows_tr, L].astype(np.float32), rep=col))
        rows.append(dict(folds=col, layer=L,
                         pear_w=r.loc['test_weighted', 'Pearson'], rmse_w=r.loc['test_weighted', 'RMSE'],
                         pear_45=r.loc['dur_45s', 'Pearson'], pear_60=r.loc['dur_60s', 'Pearson']))
S = pd.DataFrame(rows).round(4)
for col in ['g250', 'g120']:
    print(col)
    print(S[S.folds == col].sort_values('pear_w', ascending=False).head(6).drop(columns='folds').to_string(index=False))
    print()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for col in ['g250', 'g120']:
    d = S[S.folds == col]
    ax[0].plot(d.layer, d.pear_w, marker='o', label=col)
    ax[1].plot(d.layer, d.pear_45, marker='o', label=f'{col} 45s')
ax[0].plot(res.layer, res.pear_w, ls='--', color='gray', label='random folds (earlier)')
ax[0].set_title('Test-weighted CV Pearson by layer'); ax[0].set_xlabel('layer'); ax[0].legend()
ax[1].set_title('45s-group CV Pearson by layer'); ax[1].set_xlabel('layer'); ax[1].legend()
plt.tight_layout(); plt.show()

avg = S.groupby('layer').pear_w.mean()
best_L = int(avg.idxmax())
print('layer chosen by mean Pearson over g250 and g120:', best_L, round(float(avg.max()), 4))

X_tr = Em[rows_tr, best_L].astype(np.float32)
X_te = Em[rows_te, best_L].astype(np.float32)
oof_g = cv_oof(X_tr, rep='g250')
print(report(oof_g))
mdl = ridge().fit(X_tr[nz], y[nz])
pred_tr = np.clip(mdl.predict(X_tr), 1, 5)
pred_te = np.clip(mdl.predict(X_te), 1, 5)
print('TRAIN RMSE (in-sample, non-zero):', round(rmse(y[nz], pred_tr[nz]), 4))

pd.DataFrame({'uid': folds.uid, 'oof_g250': oof_g}).to_csv(f'{WORK}/oof_wavlm_g250.csv', index=False)
pd.DataFrame({'filename': test.filename, 'pred': pred_te}).to_csv(f'{WORK}/test_wavlm_L{best_L}.csv', index=False)
print('saved oof and test predictions for stacking')

In [ ]:
m = nz
a, b = np.polyfit(oof_g[m], y[m], 1, w=np.sqrt(w[m]))     # y approx a * oof + b, weighted like the test mix
print('calibration slope, intercept:', round(a, 3), round(b, 3))

cal_oof = np.clip(a * oof_g + b, 1, 5)
before = report(oof_g)
after = report(cal_oof)
print('OOF (g250) before calibration:')
print(before)
print('OOF (g250) after calibration:')
print(after)
d_rmse = before.loc['test_weighted', 'RMSE'] - after.loc['test_weighted', 'RMSE']
print('expected change in test-weighted RMSE (positive = better):', round(d_rmse, 4))

pred_cal = np.clip(a * pred_te + b, 1, 5)
print('Pearson between raw and calibrated test predictions:', round(float(np.corrcoef(pred_te, pred_cal)[0, 1]), 5))
pd.DataFrame({'filename': test.filename, 'label': pred_cal}).to_csv(f'{WORK}/submission_wavlm_calibrated.csv', index=False)
print('saved submission_wavlm_calibrated.csv')

In [ ]:
bias = oof_g - y
print('OOF bias (prediction minus label), g250, non-zero:')
for g in ['dur_45s', 'dur_60s']:
    m = nz & (grp == g)
    print(f'  {g}: mean label {y[m].mean():.3f} | mean OOF pred {oof_g[m].mean():.3f} | bias {bias[m].mean():+.3f}')
print('  test-weighted bias:', round(float(np.average(bias[nz], weights=w[nz])), 3))

print('\nTest predictions (raw model) by duration group:')
for g in ['dur_45s', 'dur_60s']:
    mt = test_grp == g
    mtr = nz & (grp == g)
    print(f'  {g}: n_test={int(mt.sum())} | mean test pred {pred_te[mt].mean():.3f} | train non-zero mean label {y[mtr].mean():.3f} | mean OOF pred {oof_g[mtr].mean():.3f}')